## Creating a RAG with ChromaDB 

#### Importing libs

In [1]:
from google import genai
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry
from google.genai import types
import chromadb
from typing import Any, Dict

import json
import os
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent.parent))
from IPython.display import Markdown

from utils.config import CLIENT, RAG_DATABASE_PATH
DB_NAME = "quarto_research_db"

### Listing all embedding models

In [2]:
for m in CLIENT.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


### Creating the embedding function

In [ ]:
# Define a helper to retry when per-minute quota is reached.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

class GeminiEmbeddingFunction(EmbeddingFunction):
    """Gemini embeddings for Chroma; implements EmbeddingFunction hooks to avoid deprecation warnings."""

    def __init__(self, document_mode: bool = True) -> None:
        self.document_mode = document_mode

    @staticmethod
    def name() -> str:
        return "google_gemini_embedding_001"

    def get_config(self) -> Dict[str, Any]:
        return {"document_mode": self.document_mode}

    @staticmethod
    def build_from_config(config: Dict[str, Any]) -> "GeminiEmbeddingFunction":
        return GeminiEmbeddingFunction(document_mode=config.get("document_mode", True))

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = CLIENT.models.embed_content(
            model="models/gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

### Creating the database with ChromaDB

In [4]:
documents = json.loads(RAG_DATABASE_PATH.read_text(encoding="utf-8"))
documents = [document['result'] for document in documents]

embed_fn = GeminiEmbeddingFunction(document_mode=True)

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

### Testing the query framework

In [ ]:
embed_fn.document_mode = False
query = "Tem muitos restaurantes na Mooca?"

result = db.query(query_texts=[query], n_results=1)
[all_passages] = result["documents"]

Markdown(all_passages[0])

Olá! Que bom que você está de olho na Mooca para investir. É um bairro com uma história riquíssima e que tem se mostrado uma excelente aposta no mercado imobiliário de São Paulo. Vamos dar uma olhada nos pontos que o tornam tão interessante:

### 1. Visão geral do bairro

A Mooca é um dos bairros mais antigos e tradicionais de São Paulo, localizado na Zona Leste da capital. Com uma forte herança italiana, que se reflete na sua cultura e gastronomia, o bairro passou por uma transformação de uma área predominantemente industrial para um vibrante centro residencial. Apesar de manter suas raízes históricas, a Mooca tem se modernizado, atraindo novos moradores e empreendimentos.

### 2. Principais fatores de valorização

*   **Localização Estratégica:** A Mooca está muito bem localizada, próxima ao centro de São Paulo e com fácil acesso a importantes vias como a Radial Leste, Marginal Tietê e Avenida do Estado. Essa conectividade é um grande atrativo para quem busca otimizar o tempo de deslocamento.
*   **Infraestrutura Urbana Completa:** O bairro oferece uma infraestrutura de serviços invejável. Conta com uma ampla rede de hospitais privados, como Sancta Maggiore, CEMA, Villa-Lobos e São Cristóvão, além de opções públicas. Na educação, a região se destaca com escolas estaduais e particulares renomadas, e se tornou um polo de ensino superior com universidades como Anhembi Morumbi e São Judas Tadeu.
*   **Comércio e Serviços Diversificados:** A Mooca tem um comércio pulsante, concentrado em vias como a Avenida Paes de Barros, Rua do Oratório e Rua da Mooca, onde se encontram bancos, escolas, correios, padarias e diversas lojas. A presença do Mooca Plaza Shopping também eleva a oferta de lazer e compras, tornando a região autossuficiente para seus moradores. Essa variedade de opções comerciais e de serviços aumenta a conveniência e, consequentemente, a atratividade para locação e moradia.
*   **Mobilidade e Acesso Facilitados:** A Mooca se beneficia de uma excelente cobertura de transporte público. O bairro conta com estações de metrô (Bresser-Mooca da Linha 3-Vermelha) e CPTM (Juventus-Mooca da Linha 10-Turquesa), além de diversas linhas de ônibus que conectam a região a outras partes da cidade. Essa facilidade de deslocamento é um diferencial enorme para a demanda por imóveis.
*   **Segurança:** A Mooca é considerada um bairro de segurança média, com uma boa porcentagem de moradores afirmando que a região é segura e possui ruas bem iluminadas. A presença de delegacias contribui para a percepção de um ambiente tranquilo para se viver e investir.

### 3. Perfil de demanda imobiliária

O perfil de moradores da Mooca tem se tornado cada vez mais heterogêneo e rejuvenescido, embora ainda preserve a forte presença de descendentes de imigrantes italianos. O custo de vida é considerado mediano, o que atrai uma diversidade de perfis, desde famílias a jovens profissionais. A busca por imóveis na Mooca é impulsionada pela qualidade de vida que o bairro oferece, aliando tradição, modernidade e um forte senso de comunidade. A região também tem visto um aumento na demanda por unidades compactas, como studios e apartamentos de menor metragem, com foco em locação, o que indica um mercado aquecido para investidores.

### 4. Conclusão sobre o potencial de investimento

A Mooca apresenta um potencial de valorização imobiliária muito promissor. O bairro está passando por uma fase de revitalização e modernização, com diversos novos empreendimentos residenciais e obras de infraestrutura em andamento. A localização estratégica, a infraestrutura desenvolvida e a qualidade de vida são fatores que impulsionam a demanda e a valorização contínua dos imóveis. A Mooca tem se destacado como um dos bairros com maior número de lançamentos na Zona Leste, atraindo o interesse de grandes incorporadoras. Para quem busca investir em ativos atrelados ao desenvolvimento da cidade, a Mooca oferece um equilíbrio entre risco controlado, potencial de retorno e demanda sustentada, especialmente em áreas próximas a estações de metrô e grandes avenidas. É, sem dúvida, uma região que merece sua atenção para um investimento sólido e com boas perspectivas de rentabilidade.

In [ ]:
documents = json.loads(RAG_DATABASE_PATH.read_text())

chunks = []

for doc in documents:

    neighborhood = doc["neighborhood"]
    markdown = doc["result"]

    sections = chunk_markdown_by_headers(markdown)

    for i, section in enumerate(sections):

        chunks.append({
            "id": f"{neighborhood}_{i}",
            "neighborhood": neighborhood,
            "section": section["section"],
            "text": f"{neighborhood} — {section['text']}"
        })